## Trapezoid

In [11]:
import math
def exponential_octile_areas(my_lambda=1):
    """
    Computes the 8 trapezoidal areas between exponential octiles.
    Octiles: Q(i/8) for i = 0,...,8 but Q(0)=0 analytically.
    """
    # Compute octiles (O_1 to O_7), O_0 is 0
    O = [0]  # O_0 = 0
    for i in range(1, 8):
        p = i / 8
        O.append(-math.log(1 - p) / my_lambda)

    # Add last point Q(1) = +infinity for exponential, but
    # for numerical consistency we approximate Q(1) using a large value.
    # Alternatively, stop at 7/8 because octiles only go to O_7.
    # We will compute 7 areas (0–1/8,...,6/8–7/8) + last interval separately.
    
    areas = []
    delta_p = 1/8

    # Compute trapezoid areas for first 7 octiles
    for i in range(0, 7):
        x_left = O[i]
        x_right = O[i+1]
        A = 0.5 * (x_left + x_right) * delta_p
        areas.append(A)

    # Last interval: p = 7/8 → 1
    # Exact Q(1) is infinite, but integral is analytic:
    # ∫_{7/8}^1 Q(p) dp = ∫_{7/8}^1 -ln(1-p) dp
    # Let u = 1-p, so du = -dp
    # When p = 7/8, u = 1/8; when p = 1, u = 0
    # ∫_{7/8}^1 -ln(1-p) dp = ∫_{1/8}^0 -ln(u) (-du) = ∫_{1/8}^0 ln(u) du = -∫_0^{1/8} ln(u) du
    # ∫ ln(u) du = u*ln(u) - u
    # So: -∫_0^{1/8} ln(u) du = -[(1/8)*ln(1/8) - 1/8 - lim_{u→0}(u*ln(u) - u)]
    # = -[(1/8)*ln(1/8) - 1/8 - 0] = -(1/8)*ln(1/8) + 1/8
    # = (1/8)*(1 - ln(1/8))

    last_area = (1/8) * (1 - math.log(1/8))
    areas.append(last_area)

    return O, areas
O, A = exponential_octile_areas(1)

print("Octiles:")
for i, v in enumerate(O):
    print(f"O[{i}] = {v}")

print("\nTrapezoid Areas (8 intervals):")
for i, a in enumerate(A):
    print(f"Area {i} [p={i/8:.3f} to {(i+1)/8:.3f}]: {a:.8f}")

# Calculate mean from areas
mean_trap = sum(A)

# Calculate MAD (H) from areas
# MAD = ∫_{0.5}^1 Q(p) dp - ∫_0^{0.5} Q(p) dp
# Areas 0-3 correspond to [0, 1/8], [1/8, 2/8], [2/8, 3/8], [3/8, 4/8] (left half: 0 to 0.5)
# Areas 4-7 correspond to [4/8, 5/8], [5/8, 6/8], [6/8, 7/8], [7/8, 1] (right half: 0.5 to 1)
I1 = sum(A[0:4])  # Left half [0, 0.5]
I2 = sum(A[4:8])  # Right half [0.5, 1]
mad_trap = I2 - I1

# True values for exponential distribution with lambda=1
true_mean = 1.0
true_mad = math.log(2)  # ln(2) ≈ 0.693147

# Calculate errors
mean_error_pct = abs((mean_trap - true_mean) / true_mean) * 100
mad_error_pct = abs((mad_trap - true_mad) / true_mad) * 100

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"Total mean (trapezoid): {mean_trap:.8f}")
print(f"MAD (trapezoid): {mad_trap:.8f}")
print(f"\nTrue mean (μ): {true_mean:.8f}")
print(f"True MAD (H): {true_mad:.8f}")
print(f"\nMean error: {abs(mean_trap - true_mean):.8f}")
print(f"Mean error %: {mean_error_pct:.4f}%")
print(f"MAD error: {abs(mad_trap - true_mad):.8f}")
print(f"MAD error %: {mad_error_pct:.4f}%")


Octiles:
O[0] = 0
O[1] = 0.13353139262452263
O[2] = 0.2876820724517809
O[3] = 0.4700036292457356
O[4] = 0.6931471805599453
O[5] = 0.9808292530117262
O[6] = 1.3862943611198906
O[7] = 2.0794415416798357

Trapezoid Areas (8 intervals):
Area 0 [p=0.000 to 0.125]: 0.00834571
Area 1 [p=0.125 to 0.250]: 0.02632584
Area 2 [p=0.250 to 0.375]: 0.04735536
Area 3 [p=0.375 to 0.500]: 0.07269693
Area 4 [p=0.500 to 0.625]: 0.10462353
Area 5 [p=0.625 to 0.750]: 0.14794523
Area 6 [p=0.750 to 0.875]: 0.21660849
Area 7 [p=0.875 to 1.000]: 0.38493019

RESULTS
Total mean (trapezoid): 1.00883128
MAD (trapezoid): 0.69938360

True mean (μ): 1.00000000
True MAD (H): 0.69314718

Mean error: 0.00883128
Mean error %: 0.8831%
MAD error: 0.00623642
MAD error %: 0.8997%


## Cubic

In [12]:
import numpy as np

# ---------------------------
# Lagrange cubic interpolation
# ---------------------------
def lagrange_cubic(x, y):
    """Return coefficients a,b,c,d of the unique cubic passing through 4 points."""
    # Fit cubic: Q(p) = a p^3 + b p^2 + c p + d
    coeffs = np.polyfit(x, y, 3)
    a, b, c, d = coeffs
    return a, b, c, d

# ---------------------------
# Integrate cubic exactly
# ---------------------------
def integrate_cubic(a, b, c, d, L, R):
    """Return integral of cubic a p^3 + b p^2 + c p + d from L to R."""
    F = lambda p: (a/4)*p**4 + (b/3)*p**3 + (c/2)*p**2 + d*p
    return F(R) - F(L)

# ---------------------------
# Your function
# ---------------------------
def cubic_octile_individual_areas(octiles):
    """
    Returns the cubic-approximation areas for each octile interval.
    A[i] = ∫_{i/8}^{(i+1)/8} Q(p) dp for i = 0,...,7.
    """
    O = np.array(octiles)

    # left cubic fit (0 to 1/2)
    p_left = np.array([1/8, 2/8, 3/8, 4/8])
    o_left = np.array([O[1], O[2], O[3], O[4]])
    a1, b1, c1, d1 = lagrange_cubic(p_left, o_left)

    # right cubic fit (1/2 to 1)
    p_right = np.array([4/8, 5/8, 6/8, 7/8])
    o_right = np.array([O[4], O[5], O[6], O[7]])
    a2, b2, c2, d2 = lagrange_cubic(p_right, o_right)

    A = []

    # left intervals: 0/8 → 4/8
    for i in range(4):
        L = i/8
        R = (i+1)/8
        Ai = integrate_cubic(a1, b1, c1, d1, L, R)
        A.append(Ai)

    # right intervals: 4/8 → 8/8
    for i in range(4, 8):
        L = i/8
        R = (i+1)/8
        Ai = integrate_cubic(a2, b2, c2, d2, L, R)
        A.append(Ai)

    return A

# ---------------------------
# Define octiles (EXAMPLE!)
# Replace with your actual values.
# Must be 9 numbers: p0..p8 or 8 numbers p1..p8
# ---------------------------
octiles = [0, 0.133, 0.287, 0.47, 0.693, 0.980, 1.386, 2.079]   # example with 8 octile points (O[0] unused)

# Compute
areas = cubic_octile_individual_areas(octiles)

# Output
print("individual cubic areas:")
for i, a in enumerate(areas):
    print(f" interval {i}/8 to {(i+1)/8}:  {a}")

print("\nsum areas =", sum(areas))


individual cubic areas:
 interval 0/8 to 0.125:  0.007994791666666782
 interval 1/8 to 0.25:  0.02600520833333343
 interval 2/8 to 0.375:  0.04695312500000008
 interval 3/8 to 0.5:  0.07221354166666669
 interval 4/8 to 0.625:  0.10419791666666689
 interval 5/8 to 0.75:  0.14576041666666595
 interval 6/8 to 0.875:  0.21269791666666782
 interval 7/8 to 1.0:  0.32601041666666575

sum areas = 0.9418333333333334


## All distribution trapezoid

In [18]:
import math
import numpy as np
from statistics import NormalDist
from scipy import stats
from scipy.integrate import quad

# Import octile computation functions (or define them if not available)
def compute_octiles_uniform():
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p)
    return res

def compute_octiles_exponential(my_lambda=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append((-math.log(1-p))/my_lambda)
    return res

def compute_octiles_laplace(my_mu=0, my_lambda=1):
    res = [0]
    O_1 = my_mu + my_lambda*math.log(2*1/8)
    O_2 = my_mu + my_lambda*math.log(2*2/8)
    O_3 = my_mu + my_lambda*math.log(2*3/8)
    O_4 = my_mu + my_lambda*math.log(2*4/8)
    O_5 = my_mu - my_lambda*math.log(2 - 2*5/8)
    O_6 = my_mu - my_lambda*math.log(2 - 2*6/8)
    O_7 = my_mu - my_lambda*math.log(2 - 2*7/8)
    res = [0, O_1, O_2, O_3, O_4, O_5, O_6, O_7]
    return res

def compute_octiles_pareto(my_alpha=1.5, my_beta=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(my_beta * ((1-p)**(-1/my_alpha)))
    return res

def compute_octiles_power(alpha=2.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p**(1/alpha))
    return res

def compute_octiles_lognormal(log_mu=0, log_sigma=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        z = NormalDist().inv_cdf(p)
        res.append(math.exp(log_mu + log_sigma * z))
    return res

def compute_octiles_normal(my_mu=0, my_sigma=1):
    res = [0]
    normal_dist = NormalDist(my_mu, my_sigma)
    for i in range(1, 8):
        p = i/8
        res.append(normal_dist.inv_cdf(p))
    return res

def compute_octiles_weibull(shape=2.0, scale=1.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(scale * ((-math.log(1-p))**(1/shape)))
    return res

# Generic trapezoid method for computing mean and MAD
def trapezoid_mean_mad(octiles, last_area_func=None):
    """
    Computes mean and MAD using trapezoid method.
    last_area_func: function to compute last area [7/8, 1] if Q(1) is infinite
    """
    O = octiles
    areas = []
    delta_p = 1/8
    
    # Compute trapezoid areas for first 7 intervals
    for i in range(0, 7):
        x_left = O[i]
        x_right = O[i+1]
        A = 0.5 * (x_left + x_right) * delta_p
        areas.append(A)
    
    # Last interval: use provided function or standard trapezoid
    if last_area_func:
        last_area = last_area_func()
    else:
        # Standard trapezoid (assumes Q(1) is finite or can be approximated)
        # For most distributions, we can use a large value or the last octile
        # For simplicity, use trapezoid with O[7] and approximate Q(1) ≈ O[7] + (O[7] - O[6])
        Q_1_approx = O[7] + (O[7] - O[6]) if len(O) > 7 else O[7]
        last_area = 0.5 * (O[7] + Q_1_approx) * delta_p
    areas.append(last_area)
    
    # Calculate mean and MAD
    mean = sum(areas)
    I1 = sum(areas[0:4])  # Left half [0, 0.5]
    I2 = sum(areas[4:8])  # Right half [0.5, 1]
    mad = I2 - I1
    
    return mean, mad

# Improved Simpson's rule method for computing mean and MAD
def simpson_mean_mad(octiles, last_area_func=None, q_func=None):
    """
    Computes mean and MAD using Simpson's 1/3 rule.
    Uses midpoint estimation for Simpson's rule on each interval.
    last_area_func: function to compute last area [7/8, 1] if Q(1) is infinite
    q_func: quantile function for better midpoint estimation (optional)
    """
    O = octiles
    areas = []
    delta_p = 1/8
    
    # Compute Simpson's rule areas for first 7 intervals
    for i in range(0, 7):
        p_left = i / 8
        p_mid = (i + 0.5) / 8
        p_right = (i + 1) / 8
        
        x_left = O[i]
        x_right = O[i+1]
        
        # Estimate midpoint value
        if q_func:
            x_mid = q_func(p_mid)
        else:
            # Linear interpolation as fallback
            x_mid = 0.5 * (x_left + x_right)
        
        # Simpson's 1/3 rule: (h/6) * [f(a) + 4f(mid) + f(b)]
        A = (delta_p / 6) * (x_left + 4 * x_mid + x_right)
        areas.append(A)
    
    # Last interval: use provided function or Simpson's rule with estimation
    if last_area_func:
        last_area = last_area_func()
    else:
        # Estimate Q(1) and midpoint for last interval
        p_mid = 15/16
        if q_func:
            Q_1_approx = q_func(0.999)  # Use 0.999 instead of 1 to avoid infinity
            x_mid = q_func(p_mid)
        else:
            # Extrapolate Q(1) and midpoint
            Q_1_approx = O[7] + (O[7] - O[6]) if len(O) > 7 else O[7]
            x_mid = 0.5 * (O[7] + Q_1_approx)
        
        last_area = (delta_p / 6) * (O[7] + 4 * x_mid + Q_1_approx)
    areas.append(last_area)
    
    # Calculate mean and MAD
    mean = sum(areas)
    I1 = sum(areas[0:4])  # Left half [0, 0.5]
    I2 = sum(areas[4:8])  # Right half [0.5, 1]
    mad = I2 - I1
    
    return mean, mad

# Cubic interpolation method (using existing implementation)
def cubic_mean_mad(octiles, last_area_func=None):
    """
    Computes mean and MAD using cubic interpolation.
    Uses cubic fits on left and right halves separately.
    """
    O = np.array(octiles)
    areas = []
    
    # Left cubic fit (0 to 1/2) - use points at 1/8, 2/8, 3/8, 4/8
    p_left = np.array([1/8, 2/8, 3/8, 4/8])
    o_left = np.array([O[1], O[2], O[3], O[4]])
    a1, b1, c1, d1 = np.polyfit(p_left, o_left, 3)
    
    # Right cubic fit (1/2 to 1) - use points at 4/8, 5/8, 6/8, 7/8
    p_right = np.array([4/8, 5/8, 6/8, 7/8])
    o_right = np.array([O[4], O[5], O[6], O[7]])
    a2, b2, c2, d2 = np.polyfit(p_right, o_right, 3)
    
    # Integrate cubic exactly: ∫ (a p^3 + b p^2 + c p + d) dp
    # = (a/4) p^4 + (b/3) p^3 + (c/2) p^2 + d p
    def integrate_cubic(a, b, c, d, L, R):
        F = lambda p: (a/4)*p**4 + (b/3)*p**3 + (c/2)*p**2 + d*p
        return F(R) - F(L)
    
    # Left intervals: 0/8 → 4/8
    for i in range(4):
        L = i/8
        R = (i+1)/8
        # For first interval [0, 1/8], use quadratic fit through (0,0), (1/8, O[1]), (2/8, O[2])
        # or use linear interpolation as simpler fallback
        if i == 0:
            # Use linear interpolation from 0 to O[1] for first interval
            # This is reasonable since we don't have a point at p=0 for the cubic fit
            A = 0.5 * (0 + O[1]) * (1/8)
        else:
            Ai = integrate_cubic(a1, b1, c1, d1, L, R)
            A = Ai
        areas.append(A)
    
    # Right intervals: 4/8 → 8/8
    for i in range(4, 8):
        L = i/8
        R = (i+1)/8
        if i == 7:
            # Last interval: use provided function or extrapolate
            if last_area_func:
                A = last_area_func()
            else:
                # Extrapolate cubic to estimate last interval
                Ai = integrate_cubic(a2, b2, c2, d2, L, R)
                A = Ai
        else:
            Ai = integrate_cubic(a2, b2, c2, d2, L, R)
            A = Ai
        areas.append(A)
    
    # Calculate mean and MAD
    mean = sum(areas)
    I1 = sum(areas[0:4])  # Left half [0, 0.5]
    I2 = sum(areas[4:8])  # Right half [0.5, 1]
    mad = I2 - I1
    
    return mean, mad

# Define distributions with their parameters and true values
STD_NORMAL = NormalDist()

def compute_true_h(q_func, eps=1e-6, num=5000):
    """Compute true MAD: ∫_{0.5}^1 Q(p) dp - ∫_0^{0.5} Q(p) dp"""
    ps_lower = np.linspace(eps, 0.5, num)
    ps_upper = np.linspace(0.5, 1 - eps, num)
    values_lower = np.array([q_func(p) for p in ps_lower])
    values_upper = np.array([q_func(p) for p in ps_upper])
    I1 = np.trapezoid(values_lower, ps_lower)
    I2 = np.trapezoid(values_upper, ps_upper)
    return I2 - I1

# Quantile functions for better midpoint estimation
def q_laplace(p, mu=0, lam=1):
    if p < 0.5:
        return mu + lam * math.log(2 * p)
    else:
        return mu - lam * math.log(2 - 2 * p)

def q_exponential(p, lam=1):
    return -math.log(1 - p) / lam

def q_uniform(p):
    return p

def q_pareto(p, alpha=2.0, beta=0.5):
    return beta * ((1 - p) ** (-1/alpha))

def q_power(p, alpha=2.0):
    return p ** (1/alpha)

def q_lognormal(p, log_mu=0, log_sigma=1):
    z = NormalDist().inv_cdf(p)
    return math.exp(log_mu + log_sigma * z)

def q_normal(p, mu=0, sigma=1):
    return NormalDist(mu, sigma).inv_cdf(p)

def q_weibull(p, shape=2.0, scale=1.0):
    return scale * ((-math.log(1 - p)) ** (1/shape))

# Better last area functions for distributions with infinite Q(1)
def pareto_last_area(alpha=2.0, beta=0.5):
    """Analytical last area for Pareto: ∫_{7/8}^1 beta*(1-p)^(-1/alpha) dp"""
    # ∫ beta*(1-p)^(-1/alpha) dp = beta * ∫ (1-p)^(-1/alpha) dp
    # Let u = 1-p, du = -dp
    # ∫ u^(-1/alpha) du = u^(1-1/alpha) / (1-1/alpha) = u^((alpha-1)/alpha) * alpha/(alpha-1)
    # At p=7/8, u=1/8; at p=1, u=0
    # ∫_{7/8}^1 = beta * alpha/(alpha-1) * [0 - (1/8)^((alpha-1)/alpha)]
    # = -beta * alpha/(alpha-1) * (1/8)^((alpha-1)/alpha)
    # But wait, this gives negative... Let me recalculate:
    # ∫_{7/8}^1 beta*(1-p)^(-1/alpha) dp
    # = beta * ∫_{u=1/8}^{u=0} u^(-1/alpha) (-du)
    # = beta * ∫_{u=0}^{u=1/8} u^(-1/alpha) du
    # = beta * [u^(1-1/alpha) / (1-1/alpha)]_{0}^{1/8}
    # = beta * alpha/(alpha-1) * (1/8)^((alpha-1)/alpha)
    return beta * alpha / (alpha - 1) * ((1/8) ** ((alpha - 1) / alpha))

distributions = [
    ("Laplace", lambda: compute_octiles_laplace(0, 1), 0, 1/math.sqrt(2), None,
     lambda p: q_laplace(p, 0, 1)),
    ("Exponential", lambda: compute_octiles_exponential(1), 1, math.log(2), 
     lambda: (1/8) * (1 - math.log(1/8)),  # Special last area for exponential
     lambda p: q_exponential(p, 1)),
    ("Uniform", lambda: compute_octiles_uniform(), 0.5, 0.25, None,
     lambda p: q_uniform(p)),
    ("Pareto", lambda: compute_octiles_pareto(2.0, 1/2), 
     2.0 * (1/2) / (2.0 - 1), 
     (1/2) * 2.0 * ((2**(1/2.0))-1)/(2.0 - 1), 
     lambda: pareto_last_area(2.0, 0.5),  # Better last area for Pareto
     lambda p: q_pareto(p, 2.0, 0.5)),
    ("Power", lambda: compute_octiles_power(2.0), 
     2.0 / (2.0 + 1), 
     compute_true_h(lambda p, a=2.0: p**(1/a)), None,
     lambda p: q_power(p, 2.0)),
    ("Log-normal", lambda: compute_octiles_lognormal(0, 1), 
     math.exp(0 + (1**2)/2), 
     compute_true_h(lambda p, m=0, s=1: math.exp(m + s * STD_NORMAL.inv_cdf(p))), None,
     lambda p: q_lognormal(p, 0, 1)),
    ("Normal", lambda: compute_octiles_normal(0, 1), 
     0, 
     compute_true_h(lambda p, nd=NormalDist(0, 1): nd.inv_cdf(p)), None,
     lambda p: q_normal(p, 0, 1)),
    ("Weibull", lambda: compute_octiles_weibull(2.0, 1.0), 
     1.0 * math.gamma(1 + 1/2.0), 
     compute_true_h(lambda p, s=2.0, b=1.0: b * ((-math.log(1-p))**(1/s)), eps=1e-6), None,
     lambda p: q_weibull(p, 2.0, 1.0)),
]

print("=" * 120)
print("COMPARISON: TRAPEZOID vs SIMPSON vs CUBIC METHODS")
print("=" * 120)
print(f"{'Distribution':<15} {'Method':<12} {'Mean':<12} {'Mean Err %':<12} {'MAD':<12} {'MAD Err %':<12}")
print("-" * 120)

results = []

for dist_name, octile_func, true_mu, true_h, last_area_func, q_func in distributions:
    octiles = octile_func()
    
    # Trapezoid method
    mean_trap, mad_trap = trapezoid_mean_mad(octiles, last_area_func)
    mean_err_trap = abs((mean_trap - true_mu) / true_mu) * 100 if abs(true_mu) > 1e-10 else abs(mean_trap - true_mu)
    mad_err_trap = abs((mad_trap - true_h) / true_h) * 100 if abs(true_h) > 1e-10 else abs(mad_trap - true_h)
    results.append((dist_name, "Trapezoid", mean_trap, mean_err_trap, mad_trap, mad_err_trap))
    
    # Simpson's method
    mean_simp, mad_simp = simpson_mean_mad(octiles, last_area_func, q_func)
    mean_err_simp = abs((mean_simp - true_mu) / true_mu) * 100 if abs(true_mu) > 1e-10 else abs(mean_simp - true_mu)
    mad_err_simp = abs((mad_simp - true_h) / true_h) * 100 if abs(true_h) > 1e-10 else abs(mad_simp - true_h)
    results.append((dist_name, "Simpson", mean_simp, mean_err_simp, mad_simp, mad_err_simp))
    
    # Cubic method
    mean_cubic, mad_cubic = cubic_mean_mad(octiles, last_area_func)
    
    mean_err_cubic = abs((mean_cubic - true_mu) / true_mu) * 100 if abs(true_mu) > 1e-10 else abs(mean_cubic - true_mu)
    mad_err_cubic = abs((mad_cubic - true_h) / true_h) * 100 if abs(true_h) > 1e-10 else abs(mad_cubic - true_h)
    results.append((dist_name, "Cubic", mean_cubic, mean_err_cubic, mad_cubic, mad_err_cubic))

# Print results
for dist_name, method, mean, mean_err, mad, mad_err in results:
    print(f"{dist_name:<15} {method:<12} {mean:<12.6f} {mean_err:<12.4f} {mad:<12.6f} {mad_err:<12.4f}")

print("=" * 120)
print("\nSUMMARY: Best method per distribution (lowest combined error)")
print("-" * 120)

# Group by distribution and find best
from collections import defaultdict
dist_results = defaultdict(list)
for dist_name, method, mean, mean_err, mad, mad_err in results:
    combined_err = mean_err + mad_err
    dist_results[dist_name].append((method, mean_err, mad_err, combined_err))

for dist_name in sorted(dist_results.keys()):
    methods = dist_results[dist_name]
    best = min(methods, key=lambda x: x[3])  # x[3] is combined_err
    print(f"{dist_name:<15} Best: {best[0]:<12} (Mean Err: {best[1]:.4f}%, MAD Err: {best[2]:.4f}%)")

print("=" * 120)


COMPARISON: TRAPEZOID vs SIMPSON vs CUBIC METHODS
Distribution    Method       Mean         Mean Err %   MAD          MAD Err %   
------------------------------------------------------------------------------------------------------------------------
Laplace         Trapezoid    0.129965     0.1300       0.721746     2.0703      
Laplace         Simpson      0.129471     0.1295       0.937366     32.5635     
Laplace         Cubic        0.152913     0.1529       0.732023     3.5236      
Exponential     Trapezoid    1.008831     0.8831       0.699384     0.8997      
Exponential     Simpson      1.000067     0.0067       0.693212     0.0094      
Exponential     Cubic        1.001418     0.1418       0.694126     0.1412      
Uniform         Trapezoid    0.500000     0.0000       0.250000     0.0000      
Uniform         Simpson      0.499979     0.0042       0.249979     0.0083      
Uniform         Cubic        0.500000     0.0000       0.250000     0.0000      
Pareto          Tra

In [6]:
## Average of quadratics for Q(0) and Q(1) with Trapezoid integration

import math
import numpy as np
from statistics import NormalDist

# Function to compute mean and MAD using average of quadratics to approximate Q(0) and Q(1)
# and trapezoid method for area calculation
def quadratic_avg_trapezoid_mean_mad(octiles, last_area_func=None):
    """
    Computes mean and MAD using:
    - Average of quadratics to approximate Q(0) and Q(1)
    - Trapezoid method for area calculation
    """
    O = np.array(octiles)
    areas = []
    delta_p = 1/8
    
    # Approximate Q(0) using average of quadratics
    # Fit quadratic to (1/8, 2/8, 3/8) and evaluate at 0
    p_q1 = np.array([1/8, 2/8, 3/8])
    o_q1 = np.array([O[1], O[2], O[3]])
    a1, b1, c1 = np.polyfit(p_q1, o_q1, 2)
    Q_0_1 = c1  # Q(0) = c1 (since p=0: a*0^2 + b*0 + c = c)
    
    # Fit quadratic to (2/8, 3/8, 4/8) and evaluate at 0
    p_q2 = np.array([2/8, 3/8, 4/8])
    o_q2 = np.array([O[2], O[3], O[4]])
    a2, b2, c2 = np.polyfit(p_q2, o_q2, 2)
    Q_0_2 = c2  # Q(0) = c2
    
    # Average the two approximations for Q(0)
    Q_0_approx = (Q_0_1 + Q_0_2) / 2
    
    # Approximate Q(1) using average of quadratics
    # Fit quadratic to (5/8, 6/8, 7/8) and evaluate at 1
    p_q3 = np.array([5/8, 6/8, 7/8])
    o_q3 = np.array([O[5], O[6], O[7]])
    a3, b3, c3 = np.polyfit(p_q3, o_q3, 2)
    Q_1_1 = a3 + b3 + c3  # Q(1) = a*1^2 + b*1 + c = a + b + c
    
    # Fit quadratic to (4/8, 5/8, 6/8) and evaluate at 1
    p_q4 = np.array([4/8, 5/8, 6/8])
    o_q4 = np.array([O[4], O[5], O[6]])
    a4, b4, c4 = np.polyfit(p_q4, o_q4, 2)
    Q_1_2 = a4 + b4 + c4  # Q(1) = a + b + c
    
    # Average the two approximations for Q(1)
    Q_1_approx = (Q_1_1 + Q_1_2) / 2
    
    # If last_area_func is provided (for distributions with infinite Q(1)), use it
    # Otherwise, use the quadratic approximation for Q(1)
    if last_area_func:
        # For last interval, use the provided function
        # But we still use Q_0_approx for first interval
        # Compute all 8 intervals using trapezoid method
        for i in range(0, 8):
            if i == 0:
                # First interval [0, 1/8]: use Q(0) approximation and O[1]
                x_left = Q_0_approx
                x_right = O[1]
                A = 0.5 * (x_left + x_right) * delta_p
            elif i == 7:
                # Last interval [7/8, 1]: use provided function
                A = last_area_func()
            else:
                # Middle intervals: use octiles directly
                x_left = O[i]
                x_right = O[i+1]
                A = 0.5 * (x_left + x_right) * delta_p
            areas.append(A)
    else:
        # Use quadratic approximations for both Q(0) and Q(1)
        for i in range(0, 8):
            if i == 0:
                # First interval [0, 1/8]: use Q(0) approximation and O[1]
                x_left = Q_0_approx
                x_right = O[1]
                A = 0.5 * (x_left + x_right) * delta_p
            elif i == 7:
                # Last interval [7/8, 1]: use O[7] and Q(1) approximation
                x_left = O[7]
                x_right = Q_1_approx
                A = 0.5 * (x_left + x_right) * delta_p
            else:
                # Middle intervals: use octiles directly
                x_left = O[i]
                x_right = O[i+1]
                A = 0.5 * (x_left + x_right) * delta_p
            areas.append(A)
    
    # Calculate mean and MAD
    mean = sum(areas)
    I1 = sum(areas[0:4])  # Left half [0, 0.5]
    I2 = sum(areas[4:8])  # Right half [0.5, 1]
    mad = I2 - I1
    
    return mean, mad

# Define distributions with their parameters and true values
STD_NORMAL = NormalDist()

def compute_true_h(q_func, eps=1e-6, num=5000):
    """Compute true MAD: ∫_{0.5}^1 Q(p) dp - ∫_0^{0.5} Q(p) dp"""
    ps_lower = np.linspace(eps, 0.5, num)
    ps_upper = np.linspace(0.5, 1 - eps, num)
    values_lower = np.array([q_func(p) for p in ps_lower])
    values_upper = np.array([q_func(p) for p in ps_upper])
    I1 = np.trapezoid(values_lower, ps_lower)
    I2 = np.trapezoid(values_upper, ps_upper)
    return I2 - I1

# Quantile functions
def q_laplace(p, mu=0, lam=1):
    if p < 0.5:
        return mu + lam * math.log(2 * p)
    else:
        return mu - lam * math.log(2 - 2 * p)

def q_exponential(p, lam=1):
    return -math.log(1 - p) / lam

def q_uniform(p):
    return p

def q_pareto(p, alpha=2.0, beta=0.5):
    return beta * ((1 - p) ** (-1/alpha))

def q_power(p, alpha=2.0):
    return p ** (1/alpha)

def q_lognormal(p, log_mu=0, log_sigma=1):
    z = NormalDist().inv_cdf(p)
    return math.exp(log_mu + log_sigma * z)

def q_normal(p, mu=0, sigma=1):
    return NormalDist(mu, sigma).inv_cdf(p)

def q_weibull(p, shape=2.0, scale=1.0):
    return scale * ((-math.log(1 - p)) ** (1/shape))

# Last area functions for distributions with infinite Q(1)
def pareto_last_area(alpha=2.0, beta=0.5):
    """Analytical last area for Pareto: ∫_{7/8}^1 beta*(1-p)^(-1/alpha) dp"""
    return beta * alpha / (alpha - 1) * ((1/8) ** ((alpha - 1) / alpha))

def compute_octiles_uniform():
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p)
    return res

def compute_octiles_exponential(my_lambda=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append((-math.log(1-p))/my_lambda)
    return res

def compute_octiles_laplace(my_mu=0, my_lambda=1):
    res = [0]
    O_1 = my_mu + my_lambda*math.log(2*1/8)
    O_2 = my_mu + my_lambda*math.log(2*2/8)
    O_3 = my_mu + my_lambda*math.log(2*3/8)
    O_4 = my_mu + my_lambda*math.log(2*4/8)
    O_5 = my_mu - my_lambda*math.log(2 - 2*5/8)
    O_6 = my_mu - my_lambda*math.log(2 - 2*6/8)
    O_7 = my_mu - my_lambda*math.log(2 - 2*7/8)
    res = [0, O_1, O_2, O_3, O_4, O_5, O_6, O_7]
    return res

def compute_octiles_pareto(my_alpha=1.5, my_beta=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(my_beta * ((1-p)**(-1/my_alpha)))
    return res

def compute_octiles_power(alpha=2.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p**(1/alpha))
    return res

def compute_octiles_lognormal(log_mu=0, log_sigma=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        z = NormalDist().inv_cdf(p)
        res.append(math.exp(log_mu + log_sigma * z))
    return res

def compute_octiles_normal(my_mu=0, my_sigma=1):
    res = [0]
    normal_dist = NormalDist(my_mu, my_sigma)
    for i in range(1, 8):
        p = i/8
        res.append(normal_dist.inv_cdf(p))
    return res

def compute_octiles_weibull(shape=2.0, scale=1.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(scale * ((-math.log(1-p))**(1/shape)))
    return res

distributions = [
    ("Laplace", lambda: compute_octiles_laplace(0, 1), 0, 1/math.sqrt(2), None),
    ("Exponential", lambda: compute_octiles_exponential(1), 1, math.log(2), 
     lambda: (1/8) * (1 - math.log(1/8))),
    ("Uniform", lambda: compute_octiles_uniform(), 0.5, 0.25, None),
    ("Pareto", lambda: compute_octiles_pareto(2.0, 1/2), 
     2.0 * (1/2) / (2.0 - 1), 
     (1/2) * 2.0 * ((2**(1/2.0))-1)/(2.0 - 1), 
     lambda: pareto_last_area(2.0, 0.5)),
    ("Power", lambda: compute_octiles_power(2.0), 
     2.0 / (2.0 + 1), 
     compute_true_h(lambda p, a=2.0: p**(1/a)), None),
    ("Log-normal", lambda: compute_octiles_lognormal(0, 1), 
     math.exp(0 + (1**2)/2), 
     compute_true_h(lambda p, m=0, s=1: math.exp(m + s * STD_NORMAL.inv_cdf(p))), None),
    ("Normal", lambda: compute_octiles_normal(0, 1), 
     0, 
     compute_true_h(lambda p, nd=NormalDist(0, 1): nd.inv_cdf(p)), None),
    ("Weibull", lambda: compute_octiles_weibull(2.0, 1.0), 
     1.0 * math.gamma(1 + 1/2.0), 
     compute_true_h(lambda p, s=2.0, b=1.0: b * ((-math.log(1-p))**(1/s)), eps=1e-6), None),
]



In [7]:
# Compute mean and MAD using average of quadratics method
print("=" * 120)
print("AVERAGE OF QUADRATICS FOR Q(0) AND Q(1) WITH TRAPEZOID INTEGRATION")
print("=" * 120)
print(f"{'Distribution':<15} {'Mean':<15} {'Mean Err %':<15} {'MAD':<15} {'MAD Err %':<15}")
print("-" * 120)

results = []

for dist_name, octile_func, true_mu, true_h, last_area_func in distributions:
    octiles = octile_func()
    
    # Average of quadratics approximation with trapezoid method
    mean_quad_avg, mad_quad_avg = quadratic_avg_trapezoid_mean_mad(octiles, last_area_func)
    
    mean_err = abs((mean_quad_avg - true_mu) / true_mu) * 100 if abs(true_mu) > 1e-10 else abs(mean_quad_avg - true_mu)
    mad_err = abs((mad_quad_avg - true_h) / true_h) * 100 if abs(true_h) > 1e-10 else abs(mad_quad_avg - true_h)
    
    results.append((dist_name, mean_quad_avg, mean_err, mad_quad_avg, mad_err))
    print(f"{dist_name:<15} {mean_quad_avg:<15.6f} {mean_err:<15.4f} {mad_quad_avg:<15.6f} {mad_err:<15.4f}")

print("=" * 120)

AVERAGE OF QUADRATICS FOR Q(0) AND Q(1) WITH TRAPEZOID INTEGRATION
Distribution    Mean            Mean Err %      MAD             MAD Err %      
------------------------------------------------------------------------------------------------------------------------
Laplace         -0.000000       0.0000          0.855815        21.0306        
Exponential     1.010489        1.0489          0.697726        0.6605         
Uniform         0.500000        0.0000          0.250000        0.0000         
Pareto          1.007613        0.7613          0.418783        1.1032         
Power           0.670225        0.5337          0.190644        2.3646         
Log-normal      1.416585        14.0798         0.879073        22.0725        
Normal          0.000000        0.0000          0.738622        7.4335         
Weibull         0.873320        1.4563          0.347163        6.1767         


In [ ]:
# Compare different averaging strategies for quadratics
# Strategy 1: Current approach - (1/8,2/8,3/8) and (2/8,3/8,4/8) for Q(0)
# Strategy 2: Using median - (1/8,2/8,4/8) and (2/8,3/8,4/8) for Q(0)
# Strategy 3: All consecutive triplets - average of all possible 3 consecutive points
# Strategy 4: Using O1, O2, median (O4) combinations

def quadratic_avg_strategy_1(octiles):
    """Current: (1/8,2/8,3/8) and (2/8,3/8,4/8) for Q(0)"""
    O = np.array(octiles)
    # Q(0) approximations
    p_q1 = np.array([1/8, 2/8, 3/8])
    o_q1 = np.array([O[1], O[2], O[3]])
    a1, b1, c1 = np.polyfit(p_q1, o_q1, 2)
    Q_0_1 = c1
    
    p_q2 = np.array([2/8, 3/8, 4/8])
    o_q2 = np.array([O[2], O[3], O[4]])
    a2, b2, c2 = np.polyfit(p_q2, o_q2, 2)
    Q_0_2 = c2
    
    Q_0 = (Q_0_1 + Q_0_2) / 2
    
    # Q(1) approximations
    p_q3 = np.array([5/8, 6/8, 7/8])
    o_q3 = np.array([O[5], O[6], O[7]])
    a3, b3, c3 = np.polyfit(p_q3, o_q3, 2)
    Q_1_1 = a3 + b3 + c3
    
    p_q4 = np.array([4/8, 5/8, 6/8])
    o_q4 = np.array([O[4], O[5], O[6]])
    a4, b4, c4 = np.polyfit(p_q4, o_q4, 2)
    Q_1_2 = a4 + b4 + c4
    
    Q_1 = (Q_1_1 + Q_1_2) / 2
    return Q_0, Q_1

def quadratic_avg_strategy_2(octiles):
    """Using median: (1/8,2/8,4/8) and (2/8,3/8,4/8) for Q(0)"""
    O = np.array(octiles)
    # Q(0) approximations - using median (O4)
    p_q1 = np.array([1/8, 2/8, 4/8])  # Includes median
    o_q1 = np.array([O[1], O[2], O[4]])
    a1, b1, c1 = np.polyfit(p_q1, o_q1, 2)
    Q_0_1 = c1
    
    p_q2 = np.array([2/8, 3/8, 4/8])  # Includes median
    o_q2 = np.array([O[2], O[3], O[4]])
    a2, b2, c2 = np.polyfit(p_q2, o_q2, 2)
    Q_0_2 = c2
    
    Q_0 = (Q_0_1 + Q_0_2) / 2
    
    # Q(1) approximations - using median (O4)
    p_q3 = np.array([4/8, 6/8, 7/8])  # Includes median
    o_q3 = np.array([O[4], O[6], O[7]])
    a3, b3, c3 = np.polyfit(p_q3, o_q3, 2)
    Q_1_1 = a3 + b3 + c3
    
    p_q4 = np.array([4/8, 5/8, 6/8])  # Includes median
    o_q4 = np.array([O[4], O[5], O[6]])
    a4, b4, c4 = np.polyfit(p_q4, o_q4, 2)
    Q_1_2 = a4 + b4 + c4
    
    Q_1 = (Q_1_1 + Q_1_2) / 2
    return Q_0, Q_1

def quadratic_avg_strategy_3(octiles):
    """All consecutive triplets: average all possible 3 consecutive points"""
    O = np.array(octiles)
    # Q(0) approximations - all consecutive triplets from left
    Q_0_vals = []
    for start_idx in range(1, 4):  # (1,2,3), (2,3,4), (3,4,5) but only first 3 make sense
        if start_idx + 2 <= 4:
            p_vals = np.array([start_idx/8, (start_idx+1)/8, (start_idx+2)/8])
            o_vals = np.array([O[start_idx], O[start_idx+1], O[start_idx+2]])
            a, b, c = np.polyfit(p_vals, o_vals, 2)
            Q_0_vals.append(c)
    
    Q_0 = np.mean(Q_0_vals) if Q_0_vals else 0
    
    # Q(1) approximations - all consecutive triplets from right
    Q_1_vals = []
    for start_idx in range(4, 6):  # (4,5,6), (5,6,7)
        if start_idx + 2 <= 7:
            p_vals = np.array([start_idx/8, (start_idx+1)/8, (start_idx+2)/8])
            o_vals = np.array([O[start_idx], O[start_idx+1], O[start_idx+2]])
            a, b, c = np.polyfit(p_vals, o_vals, 2)
            Q_1_vals.append(a + b + c)
    
    Q_1 = np.mean(Q_1_vals) if Q_1_vals else 0
    return Q_0, Q_1

def quadratic_avg_strategy_4(octiles):
    """Using O1, O2, median (O4) combinations"""
    O = np.array(octiles)
    # Q(0) approximations - combinations with O1, O2, O4
    p_q1 = np.array([1/8, 2/8, 4/8])  # O1, O2, median
    o_q1 = np.array([O[1], O[2], O[4]])
    a1, b1, c1 = np.polyfit(p_q1, o_q1, 2)
    Q_0_1 = c1
    
    p_q2 = np.array([1/8, 4/8, 2/8])  # Same points, different order (should give same result)
    o_q2 = np.array([O[1], O[4], O[2]])
    a2, b2, c2 = np.polyfit(p_q2, o_q2, 2)
    Q_0_2 = c2
    
    Q_0 = (Q_0_1 + Q_0_2) / 2
    
    # Q(1) approximations - combinations with O4, O6, O7
    p_q3 = np.array([4/8, 6/8, 7/8])  # median, O6, O7
    o_q3 = np.array([O[4], O[6], O[7]])
    a3, b3, c3 = np.polyfit(p_q3, o_q3, 2)
    Q_1_1 = a3 + b3 + c3
    
    p_q4 = np.array([4/8, 5/8, 7/8])  # median, O5, O7
    o_q4 = np.array([O[4], O[5], O[7]])
    a4, b4, c4 = np.polyfit(p_q4, o_q4, 2)
    Q_1_2 = a4 + b4 + c4
    
    Q_1 = (Q_1_1 + Q_1_2) / 2
    return Q_0, Q_1

# Test on a sample distribution (Exponential)
test_octiles = compute_octiles_exponential(1)
print("=" * 120)
print("COMPARISON OF DIFFERENT QUADRATIC AVERAGING STRATEGIES")
print("=" * 120)
print(f"{'Strategy':<30} {'Q(0) Approx':<20} {'Q(1) Approx':<20}")
print("-" * 120)

Q0_1, Q1_1 = quadratic_avg_strategy_1(test_octiles)
print(f"{'Strategy 1: (1/8,2/8,3/8) & (2/8,3/8,4/8)':<30} {Q0_1:<20.6f} {Q1_1:<20.6f}")

Q0_2, Q1_2 = quadratic_avg_strategy_2(test_octiles)
print(f"{'Strategy 2: Using median (O4)':<30} {Q0_2:<20.6f} {Q1_2:<20.6f}")

Q0_3, Q1_3 = quadratic_avg_strategy_3(test_octiles)
print(f"{'Strategy 3: All consecutive triplets':<30} {Q0_3:<20.6f} {Q1_3:<20.6f}")

Q0_4, Q1_4 = quadratic_avg_strategy_4(test_octiles)
print(f"{'Strategy 4: O1,O2,median combinations':<30} {Q0_4:<20.6f} {Q1_4:<20.6f}")

print("\nTrue values for Exponential(λ=1):")
print(f"Q(0) = 0.0, Q(1) = +∞ (but we approximate)")
print("=" * 120)

In [ ]:
# ============================================================================
# AREAS TO LEFT AND RIGHT OF QUADRATIC-APPROXIMATED ENDPOINTS
# ============================================================================
# This computes F(O_0^*) = P(X < O_0^*) and 1-F(O_8^*) = P(X > O_8^*)
# for all distributions using QUADRATIC approximations instead of cubic
# Quadratic: C_L(0) = 3O_1 - 3O_2 + O_3, C_R(1) = O_5 - 3O_6 + 3O_7

import math
import numpy as np
from statistics import NormalDist

STD_NORMAL = NormalDist()

# CDF functions (same as before)
def cdf_laplace(x, mu=0, lam=1):
    if x < mu:
        return 0.5 * math.exp((x - mu) / lam)
    else:
        return 1 - 0.5 * math.exp(-(x - mu) / lam)

def cdf_exponential(x, lam=1):
    if x < 0:
        return 0.0
    return 1 - math.exp(-lam * x)

def cdf_uniform(x):
    if x < 0:
        return 0.0
    elif x > 1:
        return 1.0
    return x

def cdf_pareto(x, alpha=2.0, beta=0.5):
    if x < beta:
        return 0.0
    return 1 - (beta / x) ** alpha

def cdf_power(x, alpha=2.0):
    if x < 0:
        return 0.0
    elif x > 1:
        return 1.0
    return x ** alpha

def cdf_lognormal(x, log_mu=0, log_sigma=1):
    if x <= 0:
        return 0.0
    z = (math.log(x) - log_mu) / log_sigma
    return STD_NORMAL.cdf(z)

def cdf_normal(x, mu=0, sigma=1):
    return NormalDist(mu, sigma).cdf(x)

def cdf_weibull(x, shape=2.0, scale=1.0):
    if x < 0:
        return 0.0
    return 1 - math.exp(-((x / scale) ** shape))

# Octile computation functions
def compute_octiles_laplace(my_mu=0, my_lambda=1):
    res = [0]
    O_1 = my_mu + my_lambda*math.log(2*1/8)
    O_2 = my_mu + my_lambda*math.log(2*2/8)
    O_3 = my_mu + my_lambda*math.log(2*3/8)
    O_4 = my_mu + my_lambda*math.log(2*4/8)
    O_5 = my_mu - my_lambda*math.log(2 - 2*5/8)
    O_6 = my_mu - my_lambda*math.log(2 - 2*6/8)
    O_7 = my_mu - my_lambda*math.log(2 - 2*7/8)
    return [0, O_1, O_2, O_3, O_4, O_5, O_6, O_7]

def compute_octiles_exponential(my_lambda=1):
    res = [0]
    for i in range(1, 8):
        p = i / 8
        res.append((-math.log(1-p))/my_lambda)
    return res

def compute_octiles_uniform():
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p)
    return res

def compute_octiles_pareto(my_alpha=2.0, my_beta=0.5):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(my_beta * ((1-p)**(-1/my_alpha)))
    return res

def compute_octiles_power(alpha=2.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p**(1/alpha))
    return res

def compute_octiles_lognormal(log_mu=0, log_sigma=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        z = NormalDist().inv_cdf(p)
        res.append(math.exp(log_mu + log_sigma * z))
    return res

def compute_octiles_normal(my_mu=0, my_sigma=1):
    res = [0]
    normal_dist = NormalDist(my_mu, my_sigma)
    for i in range(1, 8):
        p = i/8
        res.append(normal_dist.inv_cdf(p))
    return res

def compute_octiles_weibull(shape=2.0, scale=1.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(scale * ((-math.log(1-p))**(1/shape)))
    return res

# Compute quadratic-approximated endpoints
def compute_quadratic_endpoints(octiles):
    """Compute O_0^* and O_8^* using quadratic interpolation."""
    O = np.array(octiles)
    
    # Left quadratic fit through (1/8, O_1), (2/8, O_2), (3/8, O_3)
    # C_L(0) = 3O_1 - 3O_2 + O_3
    O0_star = 3*O[1] - 3*O[2] + O[3]
    
    # Right quadratic fit through (5/8, O_5), (6/8, O_6), (7/8, O_7)
    # C_R(1) = O_5 - 3O_6 + 3O_7
    O8_star = O[5] - 3*O[6] + 3*O[7]
    
    return O0_star, O8_star

# Define distributions with their parameters and CDF functions
distributions = [
    ("Laplace", lambda: compute_octiles_laplace(0, 1), 
     lambda x: cdf_laplace(x, 0, 1)),
    ("Exponential", lambda: compute_octiles_exponential(1), 
     lambda x: cdf_exponential(x, 1)),
    ("Uniform", lambda: compute_octiles_uniform(), 
     lambda x: cdf_uniform(x)),
    ("Pareto", lambda: compute_octiles_pareto(2.0, 0.5), 
     lambda x: cdf_pareto(x, 2.0, 0.5)),
    ("Power", lambda: compute_octiles_power(2.0), 
     lambda x: cdf_power(x, 2.0)),
    ("Log-normal", lambda: compute_octiles_lognormal(0, 1), 
     lambda x: cdf_lognormal(x, 0, 1)),
    ("Normal", lambda: compute_octiles_normal(0, 1), 
     lambda x: cdf_normal(x, 0, 1)),
    ("Weibull", lambda: compute_octiles_weibull(2.0, 1.0), 
     lambda x: cdf_weibull(x, 2.0, 1.0)),
]

print("=" * 120)
print("AREAS TO LEFT AND RIGHT OF QUADRATIC-APPROXIMATED ENDPOINTS")
print("=" * 120)
print(f"{'Distribution':<15} {'O_0^*':<15} {'F(O_0^*)':<15} {'O_8^*':<15} {'1-F(O_8^*)':<15}")
print("-" * 120)

results = []

for dist_name, octile_func, cdf_func in distributions:
    octiles = octile_func()
    O0_star, O8_star = compute_quadratic_endpoints(octiles)
    
    # Compute CDF values
    F_O0 = cdf_func(O0_star)
    F_O8 = cdf_func(O8_star)
    area_right = 1 - F_O8
    
    results.append((dist_name, O0_star, F_O0, O8_star, area_right))
    
    print(f"{dist_name:<15} {O0_star:<15.6f} {F_O0:<15.6e} {O8_star:<15.6f} {area_right:<15.6e}")

print("=" * 120)print("\nNote:")
print("- F(O_0^*) = P(X < O_0^*) is the probability mass to the LEFT of the quadratic-approximated Q(0)")
print("- 1-F(O_8^*) = P(X > O_8^*) is the probability mass to the RIGHT of the quadratic-approximated Q(1)")
print("- Quadratic formulas: O_0^* = 3O_1 - 3O_2 + O_3, O_8^* = O_5 - 3O_6 + 3O_7")
print("- For exact endpoints: F(Q(0)) = 0 and 1-F(Q(1)) = 0")
print("=" * 120)



In [ ]:
# ============================================================================
# AREAS TO LEFT AND RIGHT OF QUADRATIC-APPROXIMATED ENDPOINTS
# ============================================================================
# This computes F(O_0^*) = P(X < O_0^*) and 1-F(O_8^*) = P(X > O_8^*)
# for all distributions using QUADRATIC approximations instead of cubic
# Quadratic: C_L(0) = 3O_1 - 3O_2 + O_3, C_R(1) = O_5 - 3O_6 + 3O_7

import math
import numpy as np
from statistics import NormalDist

STD_NORMAL = NormalDist()

# CDF functions (same as before)
def cdf_laplace(x, mu=0, lam=1):
    if x < mu:
        return 0.5 * math.exp((x - mu) / lam)
    else:
        return 1 - 0.5 * math.exp(-(x - mu) / lam)

def cdf_exponential(x, lam=1):
    if x < 0:
        return 0.0
    return 1 - math.exp(-lam * x)

def cdf_uniform(x):
    if x < 0:
        return 0.0
    elif x > 1:
        return 1.0
    return x

def cdf_pareto(x, alpha=2.0, beta=0.5):
    if x < beta:
        return 0.0
    return 1 - (beta / x) ** alpha

def cdf_power(x, alpha=2.0):
    if x < 0:
        return 0.0
    elif x > 1:
        return 1.0
    return x ** alpha

def cdf_lognormal(x, log_mu=0, log_sigma=1):
    if x <= 0:
        return 0.0
    z = (math.log(x) - log_mu) / log_sigma
    return STD_NORMAL.cdf(z)

def cdf_normal(x, mu=0, sigma=1):
    return NormalDist(mu, sigma).cdf(x)

def cdf_weibull(x, shape=2.0, scale=1.0):
    if x < 0:
        return 0.0
    return 1 - math.exp(-((x / scale) ** shape))

# Octile computation functions
def compute_octiles_laplace(my_mu=0, my_lambda=1):
    res = [0]
    O_1 = my_mu + my_lambda*math.log(2*1/8)
    O_2 = my_mu + my_lambda*math.log(2*2/8)
    O_3 = my_mu + my_lambda*math.log(2*3/8)
    O_4 = my_mu + my_lambda*math.log(2*4/8)
    O_5 = my_mu - my_lambda*math.log(2 - 2*5/8)
    O_6 = my_mu - my_lambda*math.log(2 - 2*6/8)
    O_7 = my_mu - my_lambda*math.log(2 - 2*7/8)
    return [0, O_1, O_2, O_3, O_4, O_5, O_6, O_7]

def compute_octiles_exponential(my_lambda=1):
    res = [0]
    for i in range(1, 8):
        p = i / 8
        res.append((-math.log(1-p))/my_lambda)
    return res

def compute_octiles_uniform():
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p)
    return res

def compute_octiles_pareto(my_alpha=2.0, my_beta=0.5):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(my_beta * ((1-p)**(-1/my_alpha)))
    return res

def compute_octiles_power(alpha=2.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p**(1/alpha))
    return res

def compute_octiles_lognormal(log_mu=0, log_sigma=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        z = NormalDist().inv_cdf(p)
        res.append(math.exp(log_mu + log_sigma * z))
    return res

def compute_octiles_normal(my_mu=0, my_sigma=1):
    res = [0]
    normal_dist = NormalDist(my_mu, my_sigma)
    for i in range(1, 8):
        p = i/8
        res.append(normal_dist.inv_cdf(p))
    return res

def compute_octiles_weibull(shape=2.0, scale=1.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(scale * ((-math.log(1-p))**(1/shape)))
    return res

# Compute quadratic-approximated endpoints
def compute_quadratic_endpoints(octiles):
    """Compute O_0^* and O_8^* using quadratic interpolation."""
    O = np.array(octiles)
    
    # Left quadratic fit through (1/8, O_1), (2/8, O_2), (3/8, O_3)
    # C_L(0) = 3O_1 - 3O_2 + O_3
    O0_star = 3*O[1] - 3*O[2] + O[3]
    
    # Right quadratic fit through (5/8, O_5), (6/8, O_6), (7/8, O_7)
    # C_R(1) = O_5 - 3O_6 + 3O_7
    O8_star = O[5] - 3*O[6] + 3*O[7]
    
    return O0_star, O8_star

# Define distributions with their parameters and CDF functions
distributions = [
    ("Laplace", lambda: compute_octiles_laplace(0, 1), 
     lambda x: cdf_laplace(x, 0, 1)),
    ("Exponential", lambda: compute_octiles_exponential(1), 
     lambda x: cdf_exponential(x, 1)),
    ("Uniform", lambda: compute_octiles_uniform(), 
     lambda x: cdf_uniform(x)),
    ("Pareto", lambda: compute_octiles_pareto(2.0, 0.5), 
     lambda x: cdf_pareto(x, 2.0, 0.5)),
    ("Power", lambda: compute_octiles_power(2.0), 
     lambda x: cdf_power(x, 2.0)),
    ("Log-normal", lambda: compute_octiles_lognormal(0, 1), 
     lambda x: cdf_lognormal(x, 0, 1)),
    ("Normal", lambda: compute_octiles_normal(0, 1), 
     lambda x: cdf_normal(x, 0, 1)),
    ("Weibull", lambda: compute_octiles_weibull(2.0, 1.0), 
     lambda x: cdf_weibull(x, 2.0, 1.0)),
]

print("=" * 120)
print("AREAS TO LEFT AND RIGHT OF QUADRATIC-APPROXIMATED ENDPOINTS")
print("=" * 120)
print(f"{'Distribution':<15} {'O_0^*':<15} {'F(O_0^*)':<15} {'O_8^*':<15} {'1-F(O_8^*)':<15}")
print("-" * 120)

results = []

for dist_name, octile_func, cdf_func in distributions:
    octiles = octile_func()
    O0_star, O8_star = compute_quadratic_endpoints(octiles)
    
    # Compute CDF values
    F_O0 = cdf_func(O0_star)
    F_O8 = cdf_func(O8_star)
    area_right = 1 - F_O8
    
    results.append((dist_name, O0_star, F_O0, O8_star, area_right))
    
    print(f"{dist_name:<15} {O0_star:<15.6f} {F_O0:<15.6e} {O8_star:<15.6f} {area_right:<15.6e}")

print("=" * 120)
print("\nNote:")
print("- F(O_0^*) = P(X < O_0^*) is the probability mass to the LEFT of the quadratic-approximated Q(0)")
print("- 1-F(O_8^*) = P(X > O_8^*) is the probability mass to the RIGHT of the quadratic-approximated Q(1)")
print("- Quadratic formulas: O_0^* = 3O_1 - 3O_2 + O_3, O_8^* = O_5 - 3O_6 + 3O_7")
print("- For exact endpoints: F(Q(0)) = 0 and 1-F(Q(1)) = 0")
print("=" * 120)


AREAS TO LEFT AND RIGHT OF QUADRATIC-APPROXIMATED ENDPOINTS
Distribution    O_0^*           F(O_0^*)        O_8^*           1-F(O_8^*)     
------------------------------------------------------------------------------------------------------------------------
Laplace         -2.367124       4.687500e-02    2.367124        4.687500e-02   
Exponential     0.007552        7.523148e-03    3.060271        4.687500e-02   
Uniform         0.000000        0.000000e+00    1.000000        0.000000e+00   
Pareto          0.503972        1.570135e-02    2.059137        5.896162e-02   
Power           0.173033        2.994028e-02    0.998736        2.525915e-03   
Log-normal      0.148467        2.823505e-02    4.964052        5.455324e-02   
Normal          -1.746218       4.038652e-02    1.746218        4.038652e-02   
Weibull         0.172746        2.940049e-02    1.784219        4.144298e-02   

Note:
- F(O_0^*) = P(X < O_0^*) is the probability mass to the LEFT of the quadratic-approximated 

## Area Check

In [2]:
# ============================================================================
# AREAS TO LEFT AND RIGHT OF CUBIC-APPROXIMATED ENDPOINTS O_0^* AND O_8^*
# ============================================================================
# This computes F(O_0^*) = P(X < O_0^*) and 1-F(O_8^*) = P(X > O_8^*)
# for all distributions, showing the probability mass excluded when using
# cubic approximations instead of true Q(0) and Q(1)

import math
import numpy as np
from statistics import NormalDist
from scipy import stats

STD_NORMAL = NormalDist()

# CDF functions for each distribution
def cdf_laplace(x, mu=0, lam=1):
    """CDF of Laplace distribution."""
    if x < mu:
        return 0.5 * math.exp((x - mu) / lam)
    else:
        return 1 - 0.5 * math.exp(-(x - mu) / lam)

def cdf_exponential(x, lam=1):
    """CDF of Exponential distribution."""
    if x < 0:
        return 0.0
    return 1 - math.exp(-lam * x)

def cdf_uniform(x):
    """CDF of Uniform(0,1) distribution."""
    if x < 0:
        return 0.0
    elif x > 1:
        return 1.0
    return x

def cdf_pareto(x, alpha=2.0, beta=0.5):
    """CDF of Pareto distribution."""
    if x < beta:
        return 0.0
    return 1 - (beta / x) ** alpha

def cdf_power(x, alpha=2.0):
    """CDF of Power distribution."""
    if x < 0:
        return 0.0
    elif x > 1:
        return 1.0
    return x ** alpha

def cdf_lognormal(x, log_mu=0, log_sigma=1):
    """CDF of Log-normal distribution."""
    if x <= 0:
        return 0.0
    z = (math.log(x) - log_mu) / log_sigma
    return STD_NORMAL.cdf(z)

def cdf_normal(x, mu=0, sigma=1):
    """CDF of Normal distribution."""
    return NormalDist(mu, sigma).cdf(x)

def cdf_weibull(x, shape=2.0, scale=1.0):
    """CDF of Weibull distribution."""
    if x < 0:
        return 0.0
    return 1 - math.exp(-((x / scale) ** shape))

# Octile computation functions (reuse from earlier cells)
def compute_octiles_laplace(my_mu=0, my_lambda=1):
    res = [0]
    O_1 = my_mu + my_lambda*math.log(2*1/8)
    O_2 = my_mu + my_lambda*math.log(2*2/8)
    O_3 = my_mu + my_lambda*math.log(2*3/8)
    O_4 = my_mu + my_lambda*math.log(2*4/8)
    O_5 = my_mu - my_lambda*math.log(2 - 2*5/8)
    O_6 = my_mu - my_lambda*math.log(2 - 2*6/8)
    O_7 = my_mu - my_lambda*math.log(2 - 2*7/8)
    return [0, O_1, O_2, O_3, O_4, O_5, O_6, O_7]

def compute_octiles_exponential(my_lambda=1):
    res = [0]
    for i in range(1, 8):
        p = i / 8
        res.append((-math.log(1-p))/my_lambda)
    return res

def compute_octiles_uniform():
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p)
    return res

def compute_octiles_pareto(my_alpha=2.0, my_beta=0.5):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(my_beta * ((1-p)**(-1/my_alpha)))
    return res

def compute_octiles_power(alpha=2.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(p**(1/alpha))
    return res

def compute_octiles_lognormal(log_mu=0, log_sigma=1):
    res = [0]
    for i in range(1, 8):
        p = i/8
        z = NormalDist().inv_cdf(p)
        res.append(math.exp(log_mu + log_sigma * z))
    return res

def compute_octiles_normal(my_mu=0, my_sigma=1):
    res = [0]
    normal_dist = NormalDist(my_mu, my_sigma)
    for i in range(1, 8):
        p = i/8
        res.append(normal_dist.inv_cdf(p))
    return res

def compute_octiles_weibull(shape=2.0, scale=1.0):
    res = [0]
    for i in range(1, 8):
        p = i/8
        res.append(scale * ((-math.log(1-p))**(1/shape)))
    return res

# Compute cubic-approximated endpoints
def compute_cubic_endpoints(octiles):
    """Compute O_0^* and O_8^* using cubic interpolation."""
    O = np.array(octiles)
    
    # Left cubic fit (0 to 1/2) - use points at 1/8, 2/8, 3/8, 4/8
    p_left = np.array([1/8, 2/8, 3/8, 4/8])
    o_left = np.array([O[1], O[2], O[3], O[4]])
    a1, b1, c1, d1 = np.polyfit(p_left, o_left, 3)
    O0_star = d1  # Q(0) approximation
    
    # Right cubic fit (1/2 to 1) - use points at 4/8, 5/8, 6/8, 7/8
    p_right = np.array([4/8, 5/8, 6/8, 7/8])
    o_right = np.array([O[4], O[5], O[6], O[7]])
    a2, b2, c2, d2 = np.polyfit(p_right, o_right, 3)
    O8_star = a2 + b2 + c2 + d2  # Q(1) approximation
    
    return O0_star, O8_star

# Define distributions with their parameters and CDF functions
distributions = [
    ("Laplace", lambda: compute_octiles_laplace(0, 1), 
     lambda x: cdf_laplace(x, 0, 1)),
    ("Exponential", lambda: compute_octiles_exponential(1), 
     lambda x: cdf_exponential(x, 1)),
    ("Uniform", lambda: compute_octiles_uniform(), 
     lambda x: cdf_uniform(x)),
    ("Pareto", lambda: compute_octiles_pareto(2.0, 0.5), 
     lambda x: cdf_pareto(x, 2.0, 0.5)),
    ("Power", lambda: compute_octiles_power(2.0), 
     lambda x: cdf_power(x, 2.0)),
    ("Log-normal", lambda: compute_octiles_lognormal(0, 1), 
     lambda x: cdf_lognormal(x, 0, 1)),
    ("Normal", lambda: compute_octiles_normal(0, 1), 
     lambda x: cdf_normal(x, 0, 1)),
    ("Weibull", lambda: compute_octiles_weibull(2.0, 1.0), 
     lambda x: cdf_weibull(x, 2.0, 1.0)),
]

print("=" * 120)
print("AREAS TO LEFT AND RIGHT OF CUBIC-APPROXIMATED ENDPOINTS")
print("=" * 120)
print(f"{'Distribution':<15} {'O_0^*':<15} {'F(O_0^*)':<15} {'O_8^*':<15} {'1-F(O_8^*)':<15}")
print("-" * 120)

results = []

for dist_name, octile_func, cdf_func in distributions:
    octiles = octile_func()
    O0_star, O8_star = compute_cubic_endpoints(octiles)
    
    # Compute CDF values
    F_O0 = cdf_func(O0_star)
    F_O8 = cdf_func(O8_star)
    area_right = 1 - F_O8
    
    results.append((dist_name, O0_star, F_O0, O8_star, area_right))
    
    print(f"{dist_name:<15} {O0_star:<15.6f} {F_O0:<15.6e} {O8_star:<15.6f} {area_right:<15.6e}")

print("=" * 120)
print("\nNote:")
print("- F(O_0^*) = P(X < O_0^*) is the probability mass to the LEFT of the cubic-approximated Q(0)")
print("- 1-F(O_8^*) = P(X > O_8^*) is the probability mass to the RIGHT of the cubic-approximated Q(1)")
print("- For exact endpoints: F(Q(0)) = 0 and 1-F(Q(1)) = 0")
print("=" * 120)


AREAS TO LEFT AND RIGHT OF CUBIC-APPROXIMATED ENDPOINTS
Distribution    O_0^*           F(O_0^*)        O_8^*           1-F(O_8^*)     
------------------------------------------------------------------------------------------------------------------------
Laplace         -2.537023       3.955078e-02    2.537023        3.955078e-02   
Exponential     -0.005100       0.000000e+00    3.230170        3.955078e-02   
Uniform         0.000000        1.731641e-15    1.000000        0.000000e+00   
Pareto          0.496704        0.000000e+00    2.215734        5.092193e-02   
Power           0.156597        2.452247e-02    1.000676        0.000000e+00   
Log-normal      0.118158        1.635057e-02    5.360021        4.657916e-02   
Normal          -1.829016       3.369857e-02    1.829016        3.369857e-02   
Weibull         0.153236        2.320761e-02    1.832566        3.479445e-02   

Note:
- F(O_0^*) = P(X < O_0^*) is the probability mass to the LEFT of the cubic-approximated Q(0)
- 1

## Polyfit check

In [1]:
# ============================================================================
# VERIFY QUADRATIC & CUBIC ENDPOINT FORMULAS (ALGEBRA + polyfit CHECK)
# ============================================================================
# This cell checks the endpoint extrapolations used in the paper:
# - Quadratic through (1/8,O1),(2/8,O2),(3/8,O3):  C_L(0)=3O1-3O2+O3
# - Quadratic through (5/8,O5),(6/8,O6),(7/8,O7):  C_R(1)=O5-3O6+3O7
# - Cubic through (1/8..4/8):                     O0* = 4O1-6O2+4O3-O4
# - Cubic through (4/8..7/8):                     O8* = -O4+4O5-6O6+4O7

from fractions import Fraction
import numpy as np


def lagrange_weights_at(p_eval, ps):
    """Return Lagrange weights w_i such that P(p_eval)=sum_i w_i * y_i for nodes ps."""
    ps = [Fraction(p).limit_denominator() for p in ps]
    p_eval = Fraction(p_eval).limit_denominator()

    ws = []
    for i, pi in enumerate(ps):
        num = Fraction(1, 1)
        den = Fraction(1, 1)
        for j, pj in enumerate(ps):
            if j == i:
                continue
            num *= (p_eval - pj)
            den *= (pi - pj)
        ws.append(num / den)
    return ws


# ---- algebraic weights (exact rationals) ----
# Quadratic left @ p=0
w_qL = lagrange_weights_at(0, [Fraction(1,8), Fraction(2,8), Fraction(3,8)])
# Quadratic right @ p=1
w_qR = lagrange_weights_at(1, [Fraction(5,8), Fraction(6,8), Fraction(7,8)])

# Cubic left @ p=0
w_cL = lagrange_weights_at(0, [Fraction(1,8), Fraction(2,8), Fraction(3,8), Fraction(4,8)])
# Cubic right @ p=1
w_cR = lagrange_weights_at(1, [Fraction(4,8), Fraction(5,8), Fraction(6,8), Fraction(7,8)])

print("Algebraic (Lagrange) weights:")
print(" Quadratic left  C_L(0)  weights on (O1,O2,O3):", w_qL)
print(" Quadratic right C_R(1)  weights on (O5,O6,O7):", w_qR)
print(" Cubic left      O0*     weights on (O1,O2,O3,O4):", w_cL)
print(" Cubic right     O8*     weights on (O4,O5,O6,O7):", w_cR)


# ---- polyfit numerical check ----
# random octiles: O[0] unused (kept for indexing convenience)
O = np.concatenate([[0.0], np.random.randn(7)])

# Quadratic left fit and eval at p=0
pL2 = np.array([1/8, 2/8, 3/8])
yL2 = np.array([O[1], O[2], O[3]])
a, b, c = np.polyfit(pL2, yL2, 2)
CL0_polyfit = c
CL0_closed = 3*O[1] - 3*O[2] + O[3]

# Quadratic right fit and eval at p=1
pR2 = np.array([5/8, 6/8, 7/8])
yR2 = np.array([O[5], O[6], O[7]])
a2, b2, c2 = np.polyfit(pR2, yR2, 2)
CR1_polyfit = a2 + b2 + c2
CR1_closed = O[5] - 3*O[6] + 3*O[7]

# Cubic left fit and eval at p=0
pL3 = np.array([1/8, 2/8, 3/8, 4/8])
yL3 = np.array([O[1], O[2], O[3], O[4]])
a3, b3, c3, d3 = np.polyfit(pL3, yL3, 3)
O0_polyfit = d3
O0_closed = 4*O[1] - 6*O[2] + 4*O[3] - O[4]

# Cubic right fit and eval at p=1
pR3 = np.array([4/8, 5/8, 6/8, 7/8])
yR3 = np.array([O[4], O[5], O[6], O[7]])
a4, b4, c4, d4 = np.polyfit(pR3, yR3, 3)
O8_polyfit = a4 + b4 + c4 + d4
O8_closed = -O[4] + 4*O[5] - 6*O[6] + 4*O[7]

print("\npolyfit check (random octiles):")
print(" Quadratic left  C_L(0):  polyfit=", CL0_polyfit, " closed=", CL0_closed, " diff=", CL0_polyfit-CL0_closed)
print(" Quadratic right C_R(1):  polyfit=", CR1_polyfit, " closed=", CR1_closed, " diff=", CR1_polyfit-CR1_closed)
print(" Cubic left      O0*:     polyfit=", O0_polyfit, " closed=", O0_closed, " diff=", O0_polyfit-O0_closed)
print(" Cubic right     O8*:     polyfit=", O8_polyfit, " closed=", O8_closed, " diff=", O8_polyfit-O8_closed)


Algebraic (Lagrange) weights:
 Quadratic left  C_L(0)  weights on (O1,O2,O3): [Fraction(3, 1), Fraction(-3, 1), Fraction(1, 1)]
 Quadratic right C_R(1)  weights on (O5,O6,O7): [Fraction(1, 1), Fraction(-3, 1), Fraction(3, 1)]
 Cubic left      O0*     weights on (O1,O2,O3,O4): [Fraction(4, 1), Fraction(-6, 1), Fraction(4, 1), Fraction(-1, 1)]
 Cubic right     O8*     weights on (O4,O5,O6,O7): [Fraction(-1, 1), Fraction(4, 1), Fraction(-6, 1), Fraction(4, 1)]

polyfit check (random octiles):
 Quadratic left  C_L(0):  polyfit= 6.465531265913785  closed= 6.465531265913782  diff= 2.6645352591003757e-15
 Quadratic right C_R(1):  polyfit= 0.28600611077642046  closed= 0.28600611077641813  diff= 2.3314683517128287e-15
 Cubic left      O0*:     polyfit= 6.914517020345019  closed= 6.914517020345036  diff= -1.687538997430238e-14
 Cubic right     O8*:     polyfit= 0.23982801329883774  closed= 0.23982801329884032  diff= -2.581268532253489e-15
